In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding,AutoModelForSequenceClassification

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

* load_dataset("glue","mrpc") → loads the MRPC task from the GLUE benchmark.
MRPC = determine whether sentence1 and sentence2 have the same meaning (paraphrase) → binary classification (0/1).

* checkpoint → which pretrained model you will use.

* AutoTokenizer → loads the tokenizer for that model (BERT uses WordPiece).

In [2]:

def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

* For each example, it takes two inputs: sentence1 and sentence2.

* truncation=True → truncates text if it exceeds the maximum length (otherwise you may get an error).

* The output usually includes: input_ids, attention_mask, (and for BERT it may also include token_type_ids).

In [3]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

* .map(..., batched=True) → tokenizes in batches, which is faster.

* The resulting dataset now contains the columns needed to feed into the model.

In [4]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [5]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [6]:
data_collator

DataCollatorWithPadding(tokenizer=BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')

## TrainingArguments: Training configuration


In [7]:
from transformers import TrainingArguments
training_args = TrainingArguments("test-trainer")

* "test-trainer" → output directory: checkpoints/logs will be saved there.

* Other hyperparameters use defaults. But in practice, for better results you usually set things like:

  * learning_rate

  * per_device_train_batch_size

  * num_train_epochs

  * evaluation_strategy (in newer versions it may be called eval_strategy — your copied doc uses eval_strategy)

  * save_strategy, logging_steps

## Model: BERT with a sequence classification head

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels = 2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


* BERT backbone + a new classification head.

* Why do you see warnings?

  * BERT’s pretraining head (MLM/NSP) is not used here → some weights are unused.

  * The classification head is new → randomly initialized.

* So fine-tuning is necessary; otherwise the head remains essentially random.

## Creating the Trainer

In [9]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class = tokenizer,

)

### What’s happening here:

* model → what will be trained

* training_args → how it will be trained

* train_dataset / eval_dataset → train/validation split

* data_collator → handles batch creation (padding, etc.)

* processing_class=tokenizer → tells Trainer which tokenizer/processor to use for processing

* If you pass a tokenizer, Trainer can automatically use DataCollatorWithPadding as the default collator.

## Running training



In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


* Runs the training loop (forward → loss → backward → optimizer step)

* Fast on GPU, slow on CPU.

* By default it reports training loss, but it won’t show accuracy/F1 unless you configure metrics.

## Evaluation


## 1) `compute_metrics()` আসলে কী?

Trainer যখন evaluation (validation) চালায়, তখন সে ডিফল্টভাবে শুধু **loss** দেখায়।
আপনি যদি accuracy, F1, precision/recall ইত্যাদি দেখতে চান, Trainer-কে একটি ফাংশন দিতে হবে:

* ইনপুট: `EvalPrediction` অবজেক্ট
  এটা একটা named tuple যার মধ্যে থাকে:

  * `predictions` → মডেলের আউটপুট (logits)
  * `label_ids` → সত্যিকারের label (ground truth)

* আউটপুট: একটা dictionary

  * key = metric নাম (string) যেমন `"accuracy"`, `"f1"`
  * value = metric ভ্যালু (float)



## 2) আগে মডেল থেকে prediction বের করা (Trainer.predict)

In [ ]:
predictions = trainer.predict(tokenized_datasets["validation"])


In [ ]:
predictions

In [ ]:
print(predictions.predictions.shape , predictions.label_ids.shape)


এর মানে:

* Validation dataset-এ মোট **408টা উদাহরণ** আছে।
* `predictions.predictions.shape = (408, 2)`
  কারণ MRPC টাস্কে **২টা ক্লাস** (0/1), তাই প্রতিটা উদাহরণের জন্য মডেল 2টা স্কোর দেয়।
* `label_ids.shape = (408,)`
  প্রতিটা উদাহরণের জন্য ১টা করে সত্যি label আছে।

> `trainer.predict()` রিটার্ন করে আরেকটা named tuple:
>
> * `predictions`
> * `label_ids`
> * `metrics` (এখানে সাধারণত loss + সময়-সংক্রান্ত মেট্রিক থাকে; পরে compute_metrics দিলে accuracy/F1-ও আসবে)

## 3) logits থেকে “final class prediction” বের করা

Transformer মডেল সাধারণত **logits** দেয়—মানে raw scores (probability না)।

`(408,2)` logits থেকে আমাদের দরকার প্রতিটা উদাহরণের জন্য **কোন ক্লাসটা জিতেছে**:

In [ ]:
import numpy as np

pred = np.argmax(predictions.predictions,axis = -1)

* `axis=-1` মানে শেষ dimension (এখানে 2) বরাবর max নাও
* ফলে `preds` হবে shape `(408,)`
* প্রতিটা মান হবে 0 বা 1

**সহজ ভাষায়:**
প্রতিটা row-তে [class0_logit, class1_logit] থাকে—যেটা বড়, সেটার index-ই prediction।

## 4) Evaluate দিয়ে accuracy/F1 হিসাব করা

Hugging Face-এর `evaluate` লাইব্রেরি দিয়ে GLUE MRPC-এর metric সহজে লোড করা যায়:

In [ ]:
import evaluate
metric = evaluate.load("glue","mrpc")
metric.compute(predictions = pred, references = predictions.label_ids)

MRPC টাস্কে GLUE বেঞ্চমার্ক সাধারণত **accuracy + F1** এই দুইটা দিয়েই স্কোর দেয়।

## 5) তাহলে `compute_metrics()` ফাংশনটা কেমন হবে?

এই লজিকটাই Trainer-এর ভেতরে চালাতে `compute_metrics()` বানাতে হয়। Conceptually:

* logits → argmax → class prediction
* তারপর evaluate.compute() দিয়ে metrics

Trainer-এ দিলে evaluation-এর সময় এই metric auto দেখাবে।


## 6) কেন ফল ভিন্ন হতে পারে?

ফল একটু এদিক-ওদিক হতে পারে কারণ:

* classification head নতুনভাবে random initialize হয়
* training randomness (seed, batch order, dropout) ইত্যাদির জন্য metric সামান্য বদলাতে পারে


## 7) “BERT paper”-এর F1 তুলনা অংশটা কী বোঝায়?


* BERT paper-এর টেবিলে base model-এর F1 ~ **88.9** রিপোর্ট করা ছিল
* আপনার রান-এ ~ **89.97** এসেছে—কিছু ক্ষেত্রে এমন হতে পারে

কিন্তু একটা জিনিস লক্ষ্য করুন:
 “we are currently using the cased model”—এই কথা **আপনার আগের সেটআপের সাথে মিল নাও খেতে পারে**, কারণ আপনি আগেই `bert-base-uncased` ব্যবহার করেছেন।
অর্থাৎ:

* আপনি যদি সত্যিই `bert-base-uncased` চালান, তাহলে “cased model বলে better result” এই ব্যাখ্যাটা আপনার ক্ষেত্রে প্রযোজ্য নাও হতে পারে।
* সম্ভবত অন্য checkpoint প্রসঙ্গে সেই লাইনটা বলেছে, বা কনটেক্সট বদলেছে।
